# Escalabilidade do MILP

Reproduz o benchmark v2 (27/10) no formato novo e serve de piloto para calibrar T: mede até que tamanho o MILP prova otimalidade dentro do tempo-limite, nas famílias aleatória e em camadas, com orçamento de 25%, 50% e 75% do custo do corte total.

O MILP é a própria referência: uma solução com otimalidade provada é sua própria referência, por isso `reference=None`. A fronteira aqui usa `certified_rate` (otimalidade provada dentro de T) em vez do gap.

Diferença em relação ao v2: o tamanho 10.240 da família aleatória foi retirado, porque era degenerado e chegou a levar 134 s com limite de 20 s.

## 1. Imports, instâncias e constantes

In [ ]:
import numpy as np
import pandas as pd

from qmfnip.evaluation import comparison_table, frontier, summarize
from qmfnip.instances import Scenario
from qmfnip.outputs import DEFAULT_OUTPUT_ROOT, ExperimentOutput
from qmfnip.playground import ExperimentConfig, run_experiment
from qmfnip.plots import plot_frontier, plot_metric
from qmfnip.solvers import MILPSolver

In [ ]:
TIME_LIMIT = 20.0
BUDGET_FRACTIONS = (0.25, 0.5, 0.75)
SEEDS = (0, 1, 2)
RANDOM_SIZES = (10, 20, 40, 80, 160, 320, 640, 1280, 2560, 5120)
LAYERED_WIDTHS = (4, 6, 8, 12, 16, 24, 32, 48, 64)
Z = 0.0
P = 1.0
LEGACY_BENCHMARK = DEFAULT_OUTPUT_ROOT / "legacy" / "bench_milp_v2" / "bench_milp_v2.csv"

In [ ]:
scenarios = {
    "random": Scenario("random", RANDOM_SIZES, BUDGET_FRACTIONS, SEEDS),
    "layered": Scenario("layered", LAYERED_WIDTHS, BUDGET_FRACTIONS, SEEDS),
}
configs = {
    family: ExperimentConfig(
        scenario=scenario, solvers=[MILPSolver()], repetitions=1, time_limit=TIME_LIMIT, reference=None
    )
    for family, scenario in scenarios.items()
}
{family: len(list(scenario.instances())) for family, scenario in scenarios.items()}

## 2. Execução dos algoritmos

Um experimento por família, cada um com sua pasta em `outputs/`. A execução completa leva da ordem de 10 minutos nesta máquina.

In [ ]:
outputs = {family: run_experiment(config) for family, config in configs.items()}
{family: output.path.name for family, output in outputs.items()}

## 3. Avaliação dos resultados

Para reavaliar sem rodar de novo, substitua `outputs` por `{"random": ExperimentOutput(<pasta>), "layered": ExperimentOutput(<pasta>)}`.

In [ ]:
runs = pd.concat([output.load_runs() for output in outputs.values()], ignore_index=True)
summary = summarize(runs, z=Z)
summary[
    [
        "family", "budget_fraction", "n_nodes", "n_arcs", "runs", "certified_rate",
        "non_degenerate_rate", "median_wall_time_s", "max_wall_time_s",
    ]
]

Fronteira n\*(MILP): maior tamanho em que todas as instâncias (P = 1,0) tiveram otimalidade provada dentro de T. `reached_largest = True` indica que o limite não foi atingido nos tamanhos testados.

In [ ]:
milp_frontier = frontier(summary, p=P, rate="certified_rate")
for family, output in outputs.items():
    family_summary = summary[summary["family"] == family]
    output.write_table("summary", family_summary)
    output.write_json(
        "frontier.json", milp_frontier[milp_frontier["family"] == family].to_dict(orient="records")
    )
milp_frontier

In [ ]:
comparison_table(summary, "median_wall_time_s", index=("family", "n_nodes", "budget_fraction"))

In [ ]:
for family, output in outputs.items():
    family_summary = summary[summary["family"] == family]
    plot_metric(family_summary, "median_wall_time_s", log_y=True, path=output.figure_path("wall_time_vs_nodes"))
    plot_metric(family_summary, "certified_rate", path=output.figure_path("certified_rate_vs_nodes"))
    plot_frontier(milp_frontier[milp_frontier["family"] == family], path=output.figure_path("frontier"))

### Comparação com o benchmark v2

As instâncias são geradas com as mesmas sementes e o mesmo gerador, então o valor ótimo de cada instância certificada nos dois benchmarks deve ser **idêntico** (com tolerância de 10⁻⁶, porque o v2 gravava o objetivo do HiGHS com ruído de ponto flutuante). O tempo pode variar, e instâncias perto do limite T podem mudar de status.

In [ ]:
legacy = pd.read_csv(LEGACY_BENCHMARK).rename(
    columns={
        "familia": "family",
        "frac": "budget_fraction",
        "n": "n_nodes",
        "seed": "instance_seed",
        "value": "legacy_value",
        "optimal": "legacy_optimal",
        "time": "legacy_time_s",
    }
)
legacy["family"] = legacy["family"].map({"aleatoria": "random", "camadas": "layered"})
keys = ["family", "budget_fraction", "n_nodes", "instance_seed"]
comparison = runs[[*keys, "flow_after_interdiction", "certified_optimal", "wall_time_s"]].merge(
    legacy[[*keys, "legacy_value", "legacy_optimal", "legacy_time_s"]], on=keys, how="inner"
)
both_certified = comparison[comparison["certified_optimal"] & comparison["legacy_optimal"]]
mismatches = both_certified[
    ~np.isclose(both_certified["flow_after_interdiction"], both_certified["legacy_value"], atol=1e-6)
]
status_changes = comparison[comparison["certified_optimal"] != comparison["legacy_optimal"]]
print(
    f"matched: {len(comparison)} | both certified: {len(both_certified)} | "
    f"value mismatches: {len(mismatches)} | status changes: {len(status_changes)}"
)
status_changes